# PySpark RDD Data Engineering Project — Travel Booking Analytics

This project uses two related input files and focuses on practical PySpark RDD transformations, joins, aggregations, filtering, sorting, deduplication, and business analysis.

**Input files:** `travelers.csv`, `bookings.csv`

**Total scenario questions:** 30

In [ ]:
from pyspark import SparkContext
import csv

sc = SparkContext.getOrCreate()

rdd1 = sc.textFile("travelers.csv").mapPartitions(lambda it: csv.reader(it)).filter(lambda x: x[0]!="traveler_id")
rdd2 = sc.textFile("bookings.csv").mapPartitions(lambda it: csv.reader(it)).filter(lambda x: x[0]!="booking_id")


## Q1. Record Profiling

Count records in both input files and inspect a sample.

In [ ]:
print(rdd1.count(), rdd2.count()); print(rdd1.take(5)); print(rdd2.take(5))

## Q2. Key-Based Join

Join the two datasets using traveler_id and traveler_id.

In [ ]:
a=rdd1.map(lambda x:(x[0],x)); b=rdd2.map(lambda x:(x[1],x)); print(a.join(b).take(20))

## Q3. Group Count

Count records by home_city.

In [ ]:
print(rdd1.map(lambda x:(x[2],1)).reduceByKey(lambda a,b:a+b).collect())

## Q4. Numeric Total

Calculate total fare from the second dataset.

In [ ]:
print(rdd2.map(lambda x:float(x[4])).sum())

## Q5. Numeric Maximum

Find the maximum fare.

In [ ]:
print(rdd2.map(lambda x:float(x[4])).max())

## Q6. Numeric Minimum

Find the minimum fare.

In [ ]:
print(rdd2.map(lambda x:float(x[4])).min())

## Q7. Numeric Average

Calculate average fare.

In [ ]:
r=rdd2.map(lambda x:(float(x[4]),1)).reduce(lambda a,b:(a[0]+b[0],a[1]+b[1])); print(r[0]/r[1])

## Q8. Category-Wise Total

Calculate total fare by destination.

In [ ]:
print(rdd2.map(lambda x:(x[2],float(x[4]))).reduceByKey(lambda a,b:a+b).sortBy(lambda x:x[1],ascending=False).collect())

## Q9. Category-Wise Average

Calculate average fare by destination.

In [ ]:
r=rdd2.map(lambda x:(x[2],(float(x[4]),1))).reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1])).mapValues(lambda x:x[0]/x[1]); print(r.collect())

## Q10. Top 10 Values

Find the ten largest fare values.

In [ ]:
print(rdd2.map(lambda x:(x[4],float(x[4]))).sortBy(lambda x:x[1],ascending=False).take(10))

## Q11. Bottom 10 Values

Find the ten smallest fare values.

In [ ]:
print(rdd2.map(lambda x:(x[4],float(x[4]))).sortBy(lambda x:x[1]).take(10))

## Q12. Duplicate Key Audit

Identify duplicate traveler_id values.

In [ ]:
print(rdd2.map(lambda x:(x[1],1)).reduceByKey(lambda a,b:a+b).filter(lambda x:x[1]>1).collect())

## Q13. Filtered Numeric

Find records where fare is above its median-like operational threshold.

In [ ]:
print(rdd2.filter(lambda x:float(x[4])>float(rdd2.map(lambda y:float(y[4])).mean())).take(30))

## Q14. Sorted Categories

Sort destination totals by descending fare.

In [ ]:
r=rdd2.map(lambda x:(x[2],float(x[4]))).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1],ascending=False).collect())

## Q15. Join and Aggregate

Join the files and calculate total fare by home_city.

In [ ]:
a=rdd1.map(lambda x:(x[0],x[2])); b=rdd2.map(lambda x:(x[1],float(x[4]))); print(b.join(a).map(lambda x:(x[1][1],x[1][0])).reduceByKey(lambda a,b:a+b).collect())

## Q16. Join Filter

After joining, keep only records whose numeric measure is above 50% of its maximum.

In [ ]:
a=rdd1.map(lambda x:(x[0],x)); b=rdd2.map(lambda x:(x[1],x)); m=rdd2.map(lambda x:float(x[2] if len(x)>2 else 0)).max(); print(b.join(a).filter(lambda x:float(x[1][0][2] if len(x[1][0])>2 else 0)>m*.5).take(20))

## Q17. Category Count After Join

Count joined records by home_city.

In [ ]:
a=rdd1.map(lambda x:(x[0],x[2])); b=rdd2.map(lambda x:(x[1],x)); print(b.join(a).map(lambda x:(x[1][1],1)).reduceByKey(lambda a,b:a+b).collect())

## Q18. Key Frequency

Find the ten most frequent join keys.

In [ ]:
print(rdd2.map(lambda x:(x[1],1)).reduceByKey(lambda a,b:a+b).sortBy(lambda x:x[1],ascending=False).take(10))

## Q19. High-Value Key Analysis

Find join keys with total numeric value greater than the overall average record value.

In [ ]:
r=rdd2.map(lambda x:(x[1],float(x[2]))).reduceByKey(lambda a,b:a+b); avg=r.values().mean(); print(r.filter(lambda x:x[1]>avg).collect())

## Q20. Partition Check

Inspect the number of partitions and demonstrate repartitioning.

In [ ]:
print('Before:',rdd2.getNumPartitions()); x=rdd2.repartition(4); print('After:',x.getNumPartitions())

## Q21. Cache Workflow

Cache the second RDD and execute two actions.

In [ ]:
rdd2.cache(); print(rdd2.count()); print(rdd2.take(3))

## Q22. Distinct Categories

List distinct values of destination.

In [ ]:
print(rdd2.map(lambda x:x[2]).distinct().collect())

## Q23. Category Pair Counts

Count combinations of destination and the first categorical field in the second dataset.

In [ ]:
idx=[i for i,c in enumerate(df2.columns) if not pd.api.types.is_numeric_dtype(df2[c]) and c not in [df2.columns[0],key2]][0]; print(rdd2.map(lambda x:((x[idx],x[{list(df2.columns).index(cat2)}]),1)).reduceByKey(lambda a,b:a+b).collect())

## Q24. Numeric Range

Count records in low, medium and high value bands.

In [ ]:
r=rdd2.map(lambda x:float(x[{list(df2.columns).index(numeric2)}])); print(r.map(lambda v:'Low' if v<1000 else ('Medium' if v<5000 else 'High')).map(lambda x:(x,1)).reduceByKey(lambda a,b:a+b).collect())

## Q25. High-Value Records

Display records where fare exceeds 5000.

In [ ]:
print(rdd2.filter(lambda x:float(x[4])>5000).take(30))

## Q26. Top Category

Find the category with the highest total fare.

In [ ]:
r=rdd2.map(lambda x:(x[2],float(x[4]))).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1],ascending=False).take(1))

## Q27. Low Category

Find the three categories with the lowest total fare.

In [ ]:
r=rdd2.map(lambda x:(x[2],float(x[4]))).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1]).take(3))

## Q28. Customer/Entity Ranking

Rank join keys by total fare.

In [ ]:
r=rdd2.map(lambda x:(x[1],float(x[4]))).reduceByKey(lambda a,b:a+b); print(r.sortBy(lambda x:x[1],ascending=False).take(20))

## Q29. Final Business Summary

Build a joined business summary with home_city, record count and total fare.

In [ ]:
a=rdd1.map(lambda x:(x[0],x[2])); b=rdd2.map(lambda x:(x[1],float(x[4]))); r=b.join(a).map(lambda x:((x[1][1]),(x[1][0],1))).reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1])); print(r.sortBy(lambda x:x[1][0],ascending=False).collect())

## End of Project

The notebook is designed to run with the two CSV files in the same folder.